# Transfer learning with a pretrained ResNet-18 on EuroSAT_RGB

Fine-tune an ImageNet-pretrained `resnet18` on **`blanchon/EuroSAT_RGB`** from the Hugging Face Hub:
27,000 Sentinel-2 satellite patches, 10 land-use classes, 64x64 RGB.

**What this notebook covers**

1. Loading the dataset from the Hub (official train/validation/test splits)
2. Loading pretrained weights and replacing the classifier head
3. Two-stage training: frozen backbone (feature extraction) -> fine-tuning
4. Loss/accuracy curves, confusion matrix, and sample predictions
5. Checkpoint save/load, 512-d feature extraction, and a flattened weight vector

**Requirements:** `torch`, `torchvision`, `datasets`, `matplotlib`, `scikit-learn`

> **Note on the data:** the notebook reads the dataset from the Hub, so `data/cifar-10-python.tar.gz`
> in this repo is not used at all. EuroSAT is 64x64, so images are upscaled to the 224x224 that
> ImageNet-pretrained weights expect.

In [ ]:
# If something is missing:
# %pip install torch torchvision datasets matplotlib scikit-learn

## 1. Setup and configuration

All data comes from `load_dataset("blanchon/EuroSAT_RGB")`, which returns a `DatasetDict` with
`train` / `validation` / `test` splits already separated. We use them as-is: no re-splitting,
so the test accuracy is genuinely held out.

On CPU, start with `IMG_SIZE = 128`, `TRAIN_FRACTION = 0.1`, `EPOCHS = 2` to check that everything runs.

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, models
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# ----------------------------- config -----------------------------
SEED            = 42
HF_DATASET      = "blanchon/EuroSAT_RGB"   # 64x64 RGB, 10 classes
IMG_SIZE        = 224               # 224 = native ResNet size; 128 is ~3x faster
BATCH_SIZE      = 64
NUM_WORKERS     = 0                 # Windows/Jupyter: keep 0
TRAIN_FRACTION  = 1.0               # e.g. 0.1 for a quick smoke test
EPOCHS          = 10
LR_HEAD         = 1e-3              # LR for the new classifier head
LR_BACKBONE     = 1e-4              # LR for the pretrained backbone (once unfrozen)
WEIGHT_DECAY    = 1e-4
LABEL_SMOOTHING = 0.1
FREEZE_BACKBONE = True              # start as a pure feature extractor
UNFREEZE_AT     = 4                 # 0-based epoch to unfreeze; None = never unfreeze
CKPT_PATH       = Path("../checkpoints/resnet18_eurosat.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = device.type == "cuda"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print("device:", device)

## 2. Data and augmentation

Pretrained ImageNet models expect **224x224** inputs normalised with the ImageNet mean/std, so we
upscale the 64x64 patches and use those statistics.

* Training: random resized crop, horizontal **and** vertical flip (overhead imagery has no canonical
  orientation, so a vertical flip is a valid label-preserving augmentation here), slight colour jitter.
* Validation/test: plain resize only.

A small wrapper turns each Hugging Face split into a normal `torch.utils.data.Dataset`, which lets a
regular `DataLoader` apply the per-split transforms.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class HFImageDataset(Dataset):
    '''Wrap a Hugging Face image split as a torch Dataset (image, label).'''

    def __init__(self, split, transform):
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        item = self.split[int(idx)]
        image = item["image"].convert("RGB")
        return self.transform(image), item["label"]


raw = load_dataset(HF_DATASET)
classes = raw["train"].features["label"].names
num_classes = len(classes)

train_full = HFImageDataset(raw["train"], train_tf)
val_ds     = HFImageDataset(raw["validation"], eval_tf)
test_ds    = HFImageDataset(raw["test"], eval_tf)

# optional quick smoke test: subsample the training split only
if TRAIN_FRACTION < 1.0:
    g = torch.Generator().manual_seed(SEED)
    n_keep = max(1, int(len(train_full) * TRAIN_FRACTION))
    keep = torch.randperm(len(train_full), generator=g)[:n_keep].tolist()
    train_ds = Subset(train_full, keep)
else:
    train_ds = train_full

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

print(f"classes ({num_classes}): {classes}")
print(f"train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")

In [ ]:
def denormalize(t):
    t = t.clone()
    for c, (m, s) in enumerate(zip(MEAN, STD)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

images, labels = next(iter(train_loader))
images, labels = images[:16], labels[:16]

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(classes[int(lab)], fontsize=8)
    ax.axis("off")
fig.suptitle("Training samples (with augmentation)", y=1.02)
plt.tight_layout()
plt.show()

## 3. Model: pretrained ResNet-18

`models.resnet18(weights=...)` downloads the ImageNet weights. We replace the final fully-connected
layer (`model.fc`, 512 -> 1000) with a new one for the 10 EuroSAT classes.

With `FREEZE_BACKBONE = True` only the new head is trained at first. This is fast and hard to
overfit, and is usually a strong baseline on small datasets.

In [ ]:
def build_model(num_classes, freeze_backbone=True):
    '''ResNet-18 pretrained on ImageNet, with a fresh classifier head.'''
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    if freeze_backbone:
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith("fc.")
    return model

model = build_model(num_classes, freeze_backbone=FREEZE_BACKBONE).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {total/1e6:.2f}M total | {trainable/1e6:.2f}M trainable")

## 4. Loss, optimiser and training loop

A few details that matter for transfer learning:

* **One optimiser for the whole run.** The param groups (head / backbone-decay / backbone-no-decay)
  are created up front. Unfreezing later only flips `requires_grad`; AdamW simply skips parameters
  with no gradient. This keeps the head's optimiser state when fine-tuning starts, which rebuilding
  the optimiser mid-run would throw away.
* **Discriminative learning rates** - the head gets `LR_HEAD`, the pretrained backbone `LR_BACKBONE`.
* **No weight decay on BatchNorm weights and biases** - only on the conv/linear weights.
* **BatchNorm while frozen** - the backbone stays in `eval()` mode so its ImageNet running statistics
  are not overwritten by our batches. After unfreezing, normal training mode is used.
* **AMP** - mixed precision is enabled automatically on CUDA.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def make_optimizer(model):
    '''Head / backbone-decay / backbone-no-decay groups; all present from the start.'''
    head = [p for n, p in model.named_parameters() if n.startswith("fc.")]
    bb_decay, bb_no_decay = [], []
    for n, p in model.named_parameters():
        if n.startswith("fc."):
            continue
        (bb_no_decay if p.ndim <= 1 else bb_decay).append(p)

    groups = [{"params": head, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY}]
    if bb_decay:
        groups.append({"params": bb_decay, "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY})
    if bb_no_decay:
        groups.append({"params": bb_no_decay, "lr": LR_BACKBONE, "weight_decay": 0.0})
    return torch.optim.AdamW(groups)

optimizer = make_optimizer(model)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

print(f"optimiser groups: {len(optimizer.param_groups)} | amp: {use_amp}")

In [ ]:
def set_backbone_trainable(model, trainable):
    for name, p in model.named_parameters():
        p.requires_grad = trainable or name.startswith("fc.")

def freeze_bn(model):
    '''Keep ImageNet BatchNorm statistics while the backbone is frozen.'''
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

def train_one_epoch(model, loader, criterion, optimizer, scaler, backbone_frozen):
    model.train()
    if backbone_frozen:
        freeze_bn(model)

    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        seen += x.size(0)

    return loss_sum / seen, correct / seen

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, seen = 0.0, 0, 0
    preds, targets = [], []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.autocast(device.type, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)

        loss_sum += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        seen += x.size(0)
        preds.append(pred.cpu())
        targets.append(y.cpu())

    return (loss_sum / seen, correct / seen,
            torch.cat(preds).numpy(), torch.cat(targets).numpy())

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
best_val_acc = -1.0
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)

backbone_frozen = FREEZE_BACKBONE

for epoch in range(EPOCHS):
    # fine-tuning stage: unfreeze in place, the optimiser and its state are reused
    if backbone_frozen and UNFREEZE_AT is not None and epoch == UNFREEZE_AT:
        set_backbone_trainable(model, True)
        backbone_frozen = False
        print(f"[epoch {epoch}] backbone unfrozen -> fine-tuning")

    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer,
                                      scaler, backbone_frozen)
    va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)

    cur_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)
    history["lr"].append(cur_lr)

    print(f"epoch {epoch + 1:02d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.3f} | "
          f"lr {cur_lr:.2e} | {time.time() - t0:.0f}s")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({"model": model.state_dict(), "classes": classes,
                    "img_size": IMG_SIZE, "val_acc": va_acc}, CKPT_PATH)
        print(f"  -> new best saved ({va_acc:.3f})")

print(f"best val accuracy: {best_val_acc:.3f}")

## 5. Training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
best_epoch = int(np.argmax(history["val_acc"])) + 1

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs, history["val_loss"], marker="o", label="val")
axes[0].set(title="Loss", xlabel="epoch", ylabel="cross-entropy")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs, history["val_acc"], marker="o", label="val")
axes[1].axvline(best_epoch, color="grey", ls="--", alpha=0.6, label=f"best ({best_epoch})")
axes[1].set(title="Accuracy", xlabel="epoch", ylabel="accuracy")
axes[1].legend()

axes[2].plot(epochs, history["lr"], marker="o", color="tab:green")
axes[2].set(title="Learning rate (head)", xlabel="epoch", ylabel="lr")
axes[2].set_yscale("log")

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("ResNet-18 transfer learning on EuroSAT_RGB", y=1.03)
plt.tight_layout()
plt.show()

## 6. Evaluate the best checkpoint

This is the official EuroSAT `test` split, which the model never saw during training or for model selection (that used the `validation` split). This is a real held-out test number.

In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model"])
model.eval()
print(f"loaded best checkpoint (val acc {checkpoint['val_acc']:.3f})")

test_loss, test_acc, y_pred, y_true = evaluate(
    model, test_loader, nn.CrossEntropyLoss())
print(f"test: loss {test_loss:.4f} | accuracy {test_acc:.3f}")

print("\n" + classification_report(y_true, y_pred, labels=range(num_classes),
                                   target_names=classes, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set(title="Confusion matrix (test)", xlabel="predicted", ylabel="true")
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(classes, fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="row-normalised")
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def show_predictions(model, loader, n=12):
    model.eval()
    images, labels = next(iter(loader))
    images = images[:n]
    preds = model(images.to(device)).argmax(1).cpu()

    cols = 6
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))

    for ax, img, true, pred in zip(axes.ravel(), images, labels, preds):
        ax.imshow(denormalize(img).permute(1, 2, 0))
        ok = int(true) == int(pred)
        ax.set_title(f"true: {classes[int(true)]}\npred: {classes[int(pred)]}",
                     fontsize=8, color="green" if ok else "crimson")
        ax.axis("off")

    for ax in axes.ravel()[len(images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_predictions(model, test_loader)

## 7. Going further

**Reload the model elsewhere** - the last cell defines `load_model(path)`, which rebuilds the
architecture and restores the saved weights.

**512-d penultimate features** (for linear probes, clustering, weight-space experiments)

```python
features, targets = extract_features(model, test_loader)   # (N, 512), (N,)
```

**Tips**

* CPU only? Set `IMG_SIZE = 128`, `TRAIN_FRACTION = 0.1`, `EPOCHS = 2` first.
* Underfitting? Unfreeze earlier (`UNFREEZE_AT = 1`), train longer, or raise `LR_BACKBONE` to `3e-4`.
* Overfitting? Keep the backbone frozen, increase `WEIGHT_DECAY`, add augmentation.
* Class imbalance? Use `WeightedRandomSampler` or pass class weights to `CrossEntropyLoss`.
* Compare with from-scratch training by using `models.resnet18(weights=None)`.
* To resume training, also save `optimizer.state_dict()`, `scheduler.state_dict()` and the epoch.

In [ ]:
# 512-d penultimate features (useful for linear probes / weight-space experiments)
@torch.no_grad()
def extract_features(model, loader):
    '''Penultimate features via a hook on avgpool: (N, 512), (N,).'''
    model.eval()
    feats, ys = [], []
    handle = model.avgpool.register_forward_hook(
        lambda m, inp, out: feats.append(out.flatten(1).cpu()))
    try:
        for x, y in loader:
            model(x.to(device))
            ys.append(y)
    finally:
        handle.remove()
    return torch.cat(feats), torch.cat(ys)

features, targets = extract_features(model, test_loader)
print("features:", tuple(features.shape), "| targets:", tuple(targets.shape))

In [ ]:
# Weight-space view: flatten every parameter into a single vector
def state_dict_to_vector(model):
    return torch.cat([p.detach().flatten().cpu() for p in model.parameters()])

w = state_dict_to_vector(model)
print("weight vector:", tuple(w.shape))

In [ ]:
def load_model(path, device=device):
    ckpt = torch.load(path, map_location=device, weights_only=True)
    m = build_model(len(ckpt["classes"]), freeze_backbone=False)
    m.load_state_dict(ckpt["model"])
    return m.to(device).eval(), ckpt["classes"]

# model2, classes2 = load_model(CKPT_PATH)